<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Finding How The Data Is Distributed**


Estimated time needed: **30** minutes


In this lab, you will work with a cleaned dataset to perform Exploratory Data Analysis (EDA). You will examine the structure of the data, visualize key variables, and analyze trends related to developer experience, tools, job satisfaction, and other important aspects.


## Objectives


In this lab you will perform the following:


- Understand the structure of the dataset.

- Perform summary statistics and data visualization.

- Identify trends in developer experience, tools, job satisfaction, and other key variables.


### Install the required libraries


In [ ]:
!pip install pandas
!pip install matplotlib
!pip install seaborn


### Step 1: Import Libraries and Load Data


- Import the `pandas`, `matplotlib.pyplot`, and `seaborn` libraries.


- You will begin with loading the dataset. You can use the pyfetch method if working on JupyterLite. Otherwise, you can use pandas' read_csv() function directly on their local machines or cloud environments.


In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

# Use a clean plotting style for all charts in the notebook.
sns.set_theme(style="whitegrid")

# Load the survey data from a local file when available.
local_files = [
    Path("survey_data.csv"),
    Path("survey_data_with_duplicate.csv"),
    Path("C9-IBM Data Analyst Capstone Project/survey_data_with_duplicate.csv"),
]
remote_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"

for file_path in local_files:
    if file_path.exists():
        data_source = file_path
        break
else:
    data_source = remote_url

# Read the dataset and preview the first few rows.
df = pd.read_csv(data_source)
pd.set_option("display.max_columns", None)
df.head()


### Step 2: Examine the Structure of the Data


- Display the column names, data types, and summary information to understand the data structure.

- Objective: Gain insights into the dataset's shape and available variables.


In [ ]:
# Check the overall size of the dataset.
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

# Show the column names and data types.
print("\nColumn names:")
print(df.columns.tolist())

dtype_summary = df.dtypes.rename("data_type").reset_index()
dtype_summary.columns = ["column", "data_type"]
display(dtype_summary.head(15))

# Review the full dataframe structure and summary statistics.
print("\nDataset info:")
df.info()

print("\nSummary statistics:")
display(df.describe(include="all").transpose().head(15))


### Step 3: Handle Missing Data


- Identify missing values in the dataset.

- Impute or remove missing values as necessary to ensure data completeness.



In [ ]:
# Work on a copy so the original dataframe stays unchanged.
df_clean = df.copy()

# Find missing values before cleaning.
missing_values = df_clean.isna().sum().sort_values(ascending=False)
print("Columns with missing values:")
display(missing_values[missing_values > 0].head(15))

# Remove duplicate rows to keep one response per record.
duplicate_rows = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
print(f"Removed {duplicate_rows} duplicate rows.")

# Fill important categorical columns with their most common response.
for column in ["Employment", "RemoteWork", "EdLevel", "Country"]:
    df_clean[column] = df_clean[column].fillna(df_clean[column].mode()[0])

# Fill language columns so split/explode analysis works smoothly later.
for column in ["LanguageHaveWorkedWith", "LanguageWantToWorkWith"]:
    df_clean[column] = df_clean[column].fillna("Not specified")

# Keep a readable version of YearsCodePro and build a numeric version for analysis.
df_clean["YearsCodePro"] = df_clean["YearsCodePro"].fillna("Not specified")

def clean_years_code_pro(value):
    if pd.isna(value) or value == "Not specified":
        return pd.NA
    if value == "Less than 1 year":
        return 0.5
    if value == "More than 50 years":
        return 51
    return float(value)

df_clean["YearsCodeProNumeric"] = pd.to_numeric(
    df_clean["YearsCodePro"].apply(clean_years_code_pro), errors="coerce"
)

# Convert JobSat to numbers and leave missing values as NaN for honest charts.
df_clean["JobSat"] = pd.to_numeric(df_clean["JobSat"], errors="coerce")

print("\nRemaining missing values in selected columns:")
print(df_clean[["Employment", "RemoteWork", "EdLevel", "Country", "JobSat", "YearsCodeProNumeric"]].isna().sum())


### Step 4: Analyze Key Columns


- Examine key columns such as `Employment`, `JobSat` (Job Satisfaction), and `YearsCodePro` (Professional Coding Experience).

- **Instruction**: Calculate the value counts for each column to understand the distribution of responses.



In [ ]:
# Count the most common responses in each key column.
employment_counts = df_clean["Employment"].value_counts(dropna=False)
job_sat_counts = df_clean["JobSat"].value_counts(dropna=False).sort_index()
years_code_counts = df_clean["YearsCodePro"].value_counts(dropna=False).head(15)

print("Employment value counts:")
display(employment_counts.head(10))

print("JobSat value counts:")
display(job_sat_counts)

print("YearsCodePro value counts:")
display(years_code_counts)


### Step 5: Visualize Job Satisfaction (Focus on JobSat)


- Create a pie chart or KDE plot to visualize the distribution of `JobSat`.

- Provide an interpretation of the plot, highlighting key trends in job satisfaction.


In [ ]:
# Plot the distribution of job satisfaction scores.
job_sat_plot = df_clean["JobSat"].dropna().astype(int).value_counts().sort_index()

plt.figure(figsize=(8, 8))
plt.pie(
    job_sat_plot,
    labels=job_sat_plot.index,
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False,
)
plt.title("Distribution of Job Satisfaction Scores")
plt.show()

print("Most responses cluster around the middle-to-high satisfaction scores, with 7, 8, and 9 appearing often. Very low satisfaction scores are present, but they make up a much smaller share of the responses.")


### Step 6: Programming Languages Analysis


- Compare the frequency of programming languages in `LanguageHaveWorkedWith` and `LanguageWantToWorkWith`.
  
- Visualize the overlap or differences using a Venn diagram or a grouped bar chart.


In [ ]:
# Split semicolon-separated language answers into individual language names.
def count_multiple_choice(series):
    return (
        series.dropna()
        .str.split(";")
        .explode()
        .str.strip()
        .value_counts()
    )

languages_worked = count_multiple_choice(df_clean["LanguageHaveWorkedWith"])
languages_wanted = count_multiple_choice(df_clean["LanguageWantToWorkWith"])

# Focus on the languages that appear most across both columns.
top_languages = (languages_worked.add(languages_wanted, fill_value=0)
                 .sort_values(ascending=False)
                 .head(10)
                 .index)

language_comparison = pd.DataFrame({
    "Have worked with": languages_worked.reindex(top_languages, fill_value=0),
    "Want to work with": languages_wanted.reindex(top_languages, fill_value=0),
}).sort_values("Have worked with")

display(language_comparison.sort_values("Have worked with", ascending=False))

language_comparison.plot(kind="barh", figsize=(10, 6))
plt.title("Programming Languages: Experience vs Interest")
plt.xlabel("Number of respondents")
plt.ylabel("Programming language")
plt.tight_layout()
plt.show()


### Step 7: Analyze Remote Work Trends


- Visualize the distribution of RemoteWork by region using a grouped bar chart or heatmap.


In [ ]:
# Group countries into broad regions for a cleaner remote-work comparison.
region_groups = {
    "North America": {
        "United States of America", "Canada", "Mexico", "Jamaica", "Costa Rica", "Cuba", "Dominican Republic",
        "Guatemala", "Honduras", "Nicaragua", "Panama", "El Salvador", "Trinidad and Tobago", "Haiti",
    },
    "South America": {
        "Brazil", "Argentina", "Colombia", "Chile", "Peru", "Uruguay", "Ecuador", "Bolivia", "Paraguay", "Venezuela, Bolivarian Republic of...",
    },
    "Europe": {
        "Germany", "United Kingdom of Great Britain and Northern Ireland", "Ukraine", "France", "Poland", "Netherlands", "Italy", "Spain", "Sweden",
        "Russian Federation", "Switzerland", "Austria", "Czech Republic", "Belgium", "Denmark", "Portugal", "Norway", "Romania", "Hungary", "Greece",
        "Finland", "Bulgaria", "Ireland", "Serbia", "Croatia", "Slovakia", "Slovenia", "Lithuania", "Latvia", "Estonia", "Belarus", "Bosnia and Herzegovina",
    },
    "Asia": {
        "India", "Israel", "Turkey", "Pakistan", "Iran, Islamic Republic of...", "China", "Indonesia", "Bangladesh", "Viet Nam", "Japan", "Philippines",
        "Singapore", "Malaysia", "Sri Lanka", "Thailand", "Nepal", "United Arab Emirates", "Saudi Arabia", "Republic of Korea", "Taiwan",
    },
    "Africa": {
        "South Africa", "Nigeria", "Egypt", "Kenya", "Morocco", "Tunisia", "Ghana", "Ethiopia", "Uganda", "Algeria", "Zimbabwe",
    },
    "Oceania": {
        "Australia", "New Zealand", "Fiji"
    },
}

region_lookup = {country: region for region, countries in region_groups.items() for country in countries}
df_clean["Region"] = df_clean["Country"].map(region_lookup).fillna("Other")

remote_by_region = pd.crosstab(df_clean["Region"], df_clean["RemoteWork"], normalize="index") * 100
display(remote_by_region.round(1))

plt.figure(figsize=(10, 6))
sns.heatmap(remote_by_region.round(1), annot=True, fmt=".1f", cmap="Blues")
plt.title("Remote Work Distribution by Region (%)")
plt.xlabel("Remote work type")
plt.ylabel("Region")
plt.tight_layout()
plt.show()


### Step 8: Correlation between Job Satisfaction and Experience


- Analyze the correlation between overall job satisfaction (`JobSat`) and `YearsCodePro`.
  
- Calculate the Pearson or Spearman correlation coefficient.


In [ ]:
# Keep only rows with both variables available for correlation analysis.
correlation_data = df_clean[["JobSat", "YearsCodeProNumeric"]].dropna()

pearson_corr = correlation_data["JobSat"].corr(correlation_data["YearsCodeProNumeric"], method="pearson")
spearman_corr = correlation_data["JobSat"].corr(correlation_data["YearsCodeProNumeric"], method="spearman")

print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"Spearman correlation: {spearman_corr:.3f}")

plt.figure(figsize=(8, 5))
sns.regplot(data=correlation_data, x="YearsCodeProNumeric", y="JobSat", scatter_kws={"alpha": 0.2}, line_kws={"color": "red"})
plt.title("Job Satisfaction vs Professional Coding Experience")
plt.xlabel("Years of professional coding experience")
plt.ylabel("Job satisfaction")
plt.tight_layout()
plt.show()


### Step 9: Cross-tabulation Analysis (Employment vs. Education Level)


- Analyze the relationship between employment status (`Employment`) and education level (`EdLevel`).

- **Instruction**: Create a cross-tabulation using `pd.crosstab()` and visualize it with a stacked bar plot if possible.


In [ ]:
# Build a full cross-tabulation, then plot the largest groups for readability.
employment_edlevel = pd.crosstab(df_clean["Employment"], df_clean["EdLevel"])
display(employment_edlevel)

# Limit the chart to the most common categories so the labels stay readable.
top_employment = df_clean["Employment"].value_counts().head(8).index
top_edlevel = df_clean["EdLevel"].value_counts().head(6).index
plot_data = employment_edlevel.loc[top_employment, top_edlevel]

plot_data.plot(kind="bar", stacked=True, figsize=(12, 7), colormap="tab20")
plt.title("Employment Status by Education Level")
plt.xlabel("Employment status")
plt.ylabel("Number of respondents")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Step 10: Export Cleaned Data


- Save the cleaned dataset to a new CSV file for further use or sharing.


In [ ]:
# Save the cleaned dataframe in the lab folder when that folder is available.
output_dir = Path("C9-IBM Data Analyst Capstone Project") if Path("C9-IBM Data Analyst Capstone Project").exists() else Path(".")
output_path = output_dir / "c9lab13-cleaned-survey-data.csv"
df_clean.to_csv(output_path, index=False)
print(f"Cleaned data saved to: {output_path.resolve()}")


### Summary:


In this lab, you practiced key skills in exploratory data analysis, including:


- Examining the structure and content of the Stack Overflow survey dataset to understand its variables and data types.

- Identifying and addressing missing data to ensure the dataset's quality and completeness.

- Summarizing and visualizing key variables such as job satisfaction, programming languages, and remote work trends.

- Analyzing relationships in the data using techniques like:
    - Comparing programming languages respondents have worked with versus those they want to work with.
      
    - Exploring remote work preferences by region.

- Investigating correlations between professional coding experience and job satisfaction.

- Performing cross-tabulations to analyze relationships between employment status and education levels.


## Authors:
Ayushi Jain


### Other Contributors:
Rav Ahuja
Lakshmi Holla
Malika


Copyright © IBM Corporation. All rights reserved.
